# Data, integration and exploratory analysis

Run after group-owned collection and preparation. Every output must be interpreted and checked by the group. No results are supplied. The rectangle is around Switzerland, not a national boundary; the nearest selected airport is not an inferred origin/destination.

In [ ]:
from contextlib import closing
from pathlib import Path
import sqlite3
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
ROOT = Path.cwd() if (Path.cwd() / "config.json").exists() else Path.cwd().parent
DB = ROOT / "data/processed/aircraft.sqlite"
if not DB.exists():
    raise RuntimeError("Collect real data and run src/prepare.py first.")
with closing(sqlite3.connect(DB)) as db:
    data = pd.read_sql_query("SELECT * FROM observations", db)
    airports = pd.read_sql_query("SELECT * FROM airports", db)
    audit = pd.read_sql_query("SELECT * FROM integration_audit", db)
data["observed_at_utc"] = pd.to_datetime(data.observed_at_utc, utc=True)
OUT = ROOT / "outputs"
(OUT / "figures").mkdir(parents=True, exist_ok=True)
(OUT / "tables").mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")


In [ ]:
from contextlib import closing
display(audit)
display(data.isna().sum().to_frame("missing_after_preparation"))
with closing(sqlite3.connect(DB)) as db:
    coverage = pd.read_sql_query("""SELECT date_utc, COUNT(*) observations,
        COUNT(DISTINCT icao24) aircraft FROM observations GROUP BY date_utc ORDER BY date_utc""", db)
    summary = pd.read_sql_query("""SELECT a.name, o.zone, COUNT(*) observations,
        COUNT(DISTINCT o.icao24) aircraft, AVG(o.velocity) mean_speed_m_s
        FROM observations o JOIN airports a ON o.airport_ident=a.ident
        GROUP BY a.name, o.zone""", db)
display(coverage); display(summary)
coverage.to_csv(OUT / "tables/date_coverage.csv", index=False)
summary.to_csv(OUT / "tables/airport_zone_summary.csv", index=False)
display(data.groupby("zone")[["velocity", "geo_altitude", "distance_km"]].describe())

In [ ]:
# Acquisition coverage includes failures and empty snapshots, not just retained rows.
manifest = pd.read_json(ROOT / "data/raw/manifest.jsonl", lines=True)
requests_log = manifest[manifest.source_url.str.contains("states/all", regex=False)].copy()
requests_log["retrieved_at_utc"] = pd.to_datetime(requests_log.retrieved_at_utc, utc=True)
# HTTP 200 alone is insufficient: schema validation must succeed and raw bytes must be saved.
requests_log["successful"] = requests_log.status.eq(200) & requests_log.path.notna()
requests_log["attempted"] = requests_log.get("outcome", pd.Series(index=requests_log.index, dtype=str)).ne("deferred")
print("Invocations:", len(requests_log), "Attempted:", requests_log.attempted.sum(), "Successful:", requests_log.successful.sum())
requests_log.set_index("retrieved_at_utc")[["attempted", "successful"]].resample("h").sum().plot(figsize=(12,4))
plt.ylabel("Collection attempts / successes"); plt.tight_layout()
plt.savefig(OUT / "figures/collection_coverage.png", dpi=160)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))
sns.histplot(data=data, x="velocity", hue="zone", element="step", stat="density", common_norm=False, ax=ax[0])
sns.histplot(data=data, x="geo_altitude", hue="zone", element="step", stat="density", common_norm=False, ax=ax[1])
ax[0].set_xlabel("Ground speed (m/s)"); ax[1].set_xlabel("Geometric altitude (m)")
fig.tight_layout(); fig.savefig(OUT / "figures/group_distributions.png", dpi=160); plt.show()
plot_sample = data.sample(min(5000,len(data)), random_state=42)
sns.scatterplot(data=plot_sample, x="geo_altitude", y="velocity", hue="zone", alpha=0.25, s=12)
plt.xlabel("Geometric altitude (m)"); plt.ylabel("Ground speed (m/s)")
plt.tight_layout(); plt.savefig(OUT / "figures/altitude_speed.png", dpi=160); plt.show()
# Check overlapping altitude support before comparing speed.
data["altitude_band_m"] = pd.cut(data.geo_altitude, bins=np.arange(-1000,21001,2000))
banded = data.groupby(["zone", "altitude_band_m"], observed=True).agg(
    observations=("icao24","size"), aircraft=("icao24","nunique"), median_speed_m_s=("velocity","median"))
display(banded)
banded.to_csv(OUT / "tables/altitude_band_comparison.csv")

In [ ]:
import folium
config = json.loads((ROOT / "config.json").read_text())
colors = {"near":"#d97706", "middle":"#64748b", "far":"#2563eb"}
m = folium.Map(location=[46.8,8.2], zoom_start=7)
for airport in airports.itertuples():
    folium.Marker([airport.latitude_deg,airport.longitude_deg], tooltip=airport.ident).add_to(m)
for row in data.sample(min(1500,len(data)), random_state=42).itertuples():
    folium.CircleMarker([row.latitude,row.longitude],radius=2,color=colors[row.zone],
        fill=True,fill_opacity=.5,tooltip=f"{row.zone}; {row.velocity:.1f} m/s; nearest selected airport {row.airport_ident}").add_to(m)
# Explicit legend so colors are interpretable outside the notebook.
from branca.element import Element
m.get_root().html.add_child(Element('<div style="position:fixed;bottom:30px;left:30px;background:white;padding:12px;z-index:9999">'
    f'<b>Proximity to selected airports</b><br><span style="color:#d97706">Near: <= {config["near_km"]} km</span><br>'
    f'<span style="color:#64748b">Middle: {config["near_km"]}-{config["far_km"]} km</span><br><span style="color:#2563eb">Far: >= {config["far_km"]} km</span><br>'
    'Sample of observations; ground speed in m/s</div>'))
m.save(str(OUT / "figures/aircraft_map.html"))
m

## Group interpretation required

Describe coverage, exclusions, counts of distinct aircraft, sample imbalance and common altitude support. Explain map sampling, repeated observations and ADS-B coverage bias. Check map legend thresholds if config changes. Add an aircraft-balanced sensitivity comparison before final conclusions.

## Optional aircraft metadata

A separate aircraft table is left-joined by normalized ICAO24. Unmatched reports stay in the analysis. Key matches do not prove current or complete metadata: the downloaded reference is old and mixes sources. Compare observation and distinct-aircraft coverage, inspect the unknown/other category, and distinguish observation frequency from aircraft frequency.

In [ ]:
from contextlib import closing
with closing(sqlite3.connect(DB)) as db:
    has_metadata = db.execute("SELECT COUNT(*) FROM sqlite_master WHERE name='aircraft_metadata'").fetchone()[0]
    if has_metadata:
        metadata_audit = pd.read_sql_query("SELECT * FROM metadata_integration_audit", db)
        types = pd.read_sql_query("""SELECT COALESCE(NULLIF(m.aircraft_typecode,''),'unknown/other') typecode,
            COALESCE(m.aircraft_category,'unknown/other') aircraft_category,
            COUNT(*) observations, COUNT(DISTINCT o.icao24) aircraft
            FROM observations o LEFT JOIN aircraft_metadata m ON o.icao24=m.icao24
            GROUP BY 1,2 ORDER BY aircraft DESC""", db)
        display(metadata_audit); display(types.head(20))
        types.to_csv(OUT / "tables/aircraft_type_coverage.csv", index=False)
    else:
        print("Optional metadata tables absent. Existing EDA is still valid; rebuild with src/prepare.py for enrichment.")
